# 13. Final evaluation, part B: fold10 contact

This is the second half of the two-part final evaluation. The ensemble was frozen in `13_final_evaluation_ensemble.ipynb` (folds 1 to 9); here the frozen system makes its **single atomic contact with fold10**, the sacred held-out test. Ensemble-freeze and fold10-contact are deliberately kept as two separate halves. See `JOURNAL.md` (the ensemble rank-vote and single fold10-contact decisions).

# 13 — fold10 final (single atomic test-set contact)

**Irreversible.** Read-only + inference only (frozen `model_raw.joblib` + predict; NO refit, NO ensemble re-decision, NO re-thresholding). Deployed model = frozen M3+M4 rank-vote (alpha=0.50, threshold=0.9969), from `models/ensemble/ensemble_config.json`. Ranking is always against the frozen folds-1-8 reference (`ref_scores_M3/M4.npy`) — fold9/fold10 are NEVER re-ranked jointly. Threshold is the frozen 0.9969, never recomputed.

**HARD STOP protocol.** PART 1 = Block 0-2 (inference on both folds + fold9 validation; fold9 is safe to look at). PART 2 = Block 3-3b-4 (fold10 evaluation + per-model cards + summary). **Run PART 1 only, paste the fold9 output, wait for confirmation, THEN run PART 2.** Once Block 3 runs, fold10 is burned — no 'let me also try', ever.

**M6/M7 note.** M6 (PTB-only, reference) and M7 (OOF-only, reference) are scored from CSVs on disk (M6 via `fix_m6_fold10_scores.py`: 6 marquette globals + 4 discovery means; M7 via `m7_fold9_fold10_scores.ipynb`, 5 seeds on folds 1-8). Both are reference rows, out of the deployed ensemble. M7 fold9 reproduction vs run7 = corr 0.94 (5-on-8-folds vs 40-OOF, expected, not a defect); M6 fold9 AP reproduces 0.789.

## Block 0 — setup + frozen artifacts

In [ ]:
# ===================== BLOCK 0 — setup + frozen artifacts =====================
# the final evaluation: single atomic test-set contact. Read-only + INFERENCE ONLY (load frozen model_raw.joblib + predict).
# NO refit anywhere. Ranking always vs the frozen folds-1-8 reference. Threshold = frozen 0.9969 (never recomputed).
import os, sys, json, time, warnings
import numpy as np, pandas as pd, joblib
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROC=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src'); MOD=os.path.join(ROOT,'models')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics')
for d in (FIG,METRICS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC); from evaluation import evaluate_standard, _boot_ci
KEY=['ecg_id','source']
CFG=json.load(open(os.path.join(MOD,'ensemble','ensemble_config.json')))
ALPHA=float(CFG['alpha_weight_on_M4']); THR=float(CFG['threshold_F1max_on_OOF_ensemble']); OOF_AP=float(CFG['oof_AP'])
refM3=np.load(os.path.join(MOD,'ensemble','ref_scores_M3.npy')); refM4=np.load(os.path.join(MOD,'ensemble','ref_scores_M4.npy'))
print('Frozen ensemble: members %s | alpha=%.2f | threshold=%.4f | OOF_AP=%.4f | ref len M3=%d M4=%d'%(
    CFG['members'],ALPHA,THR,OOF_AP,len(refM3),len(refM4)))
def pctile(x,ref): return np.searchsorted(ref,np.asarray(x),side='right')/len(ref)   # percentile vs FROZEN reference
meta=pd.read_csv(os.path.join(PROC,'metadata_combined.csv'),dtype={'ecg_id':str})
for f in (9,10): print('  fold%d: %d ECG / %d WPW'%(f,int((meta.fold==f).sum()),int(((meta.fold==f)&(meta.label==1)).sum())))
def is_ptb(s): return str(s).lower().startswith('ptb')
# model registry (audited): name -> (model_raw, feature_source, matrix, platt, ptb_only)
# NOTE: M6 is NOT here -> it is loaded from CSV in Block 1 (its 10 features span two files; scored by fix_m6_fold10_scores.py).
REG={
 'M1':('M1_neurokit/m1_combined_model_raw.joblib','M1_neurokit/m1_combined_features.csv','m1_features.csv','M1_neurokit/m1_combined_platt.joblib',False),
 'M2':('M2_statistical/m2_xgb_raw.joblib','M2_statistical/m2_selected_features.csv','m2_features.csv','M2_statistical/m2_platt_calibrator.joblib',False),
 'M3':('M3_wavelet/m3_combined_model_raw.joblib','M3_wavelet/m3_combined_features.csv','m3_features.csv','M3_wavelet/m3_combined_platt.joblib',False),
 'M4':('M4_medianbeat/m4_combined_wavelet_env_model_raw.joblib','M4_medianbeat/m4_combined_wavelet_env_features.csv','m4_features_wavelet_env.csv','M4_medianbeat/m4_combined_wavelet_env_platt.joblib',False),
 'M5v2':('M5_spatial/m5v2_combined_model_raw.joblib','M5_spatial/m5v2_combined_features.csv','m5v2_features.csv','M5_spatial/m5v2_combined_platt.joblib',False),
 'best':('best_model/bestmodel_model_raw.joblib','best_model/bestmodel_features.csv','bestmodel_pool.csv','best_model/bestmodel_platt.joblib',False),
}
REF_F9={'M1':0.665,'M2':0.399,'M3':0.654,'M4':0.837,'M5v2':0.443,'best':0.790,'M6':0.789,'M7':0.834}   # known fold9 AP (sanity)
def feat_list(fsrc):
    if fsrc.endswith('.json'): return list(json.load(open(os.path.join(MOD,fsrc)))['features'])
    d=pd.read_csv(os.path.join(MOD,fsrc)); return d[('feature' if 'feature' in d.columns else d.columns[0])].astype(str).tolist()
def build_oof_ens():
    m3=pd.read_csv(os.path.join(PROC,'m3_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','source','label','proba_raw']].rename(columns={'proba_raw':'m3'})
    m4=pd.read_csv(os.path.join(PROC,'m4_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','source','proba_raw']].rename(columns={'proba_raw':'m4'})
    d=m3.merge(m4,on=KEY,how='inner')
    ens=ALPHA*pctile(d.m4.values,refM4)+(1-ALPHA)*pctile(d.m3.values,refM3)
    return d.label.values.astype(int), ens
def reload_scores():
    return pd.read_csv(os.path.join(PROC,'fold10_eval_cache.csv'),dtype={'ecg_id':str})
def comp_table(Sf,fold):
    rows=[]
    for name in ['M1','M2','M3','M4','M5v2','best','M6','M7']:
        col=name+'_raw'
        if col not in Sf.columns: continue
        sub=Sf.dropna(subset=[col])
        if name=='M6': sub=sub[sub.source.map(is_ptb)]
        if sub[col].notna().sum()>=5 and sub.label.sum()>=1:
            rows.append(dict(model=name,n=len(sub),nWPW=int(sub.label.sum()),
                             AP=round(float(average_precision_score(sub.label,sub[col])),4),
                             AUC=round(float(roc_auc_score(sub.label,sub[col])),4)))
    return pd.DataFrame(rows)
def ens_of(Sf):   # ensemble rank-vote score for a fold subset (rank vs FROZEN reference)
    return ALPHA*pctile(Sf['M4_raw'].values,refM4)+(1-ALPHA)*pctile(Sf['M3_raw'].values,refM3)
print('Block 0 OK.')

## Block 1 — inference of frozen models on fold9 AND fold10 (no refit) — PART 1

In [ ]:
# ===================== BLOCK 1 — inference of frozen models on fold9 AND fold10 (no refit) =====================
# Reads fold9/fold10 rows from the big feature matrices (chunked, usecols) -> predict raw with frozen joblib.
# Computes fold10 SCORES here (inference) but does NOT evaluate fold10 (metric = Block 3). Per-model cache = resumable.
CACHE=os.path.join(PROC,'fold10_eval_cache.csv')
def load_fold910(matrix,feats):
    use=set(['ecg_id','source','fold','label']+list(feats)); parts=[]
    for ch in pd.read_csv(os.path.join(PROC,matrix),usecols=lambda c:c in use,dtype={'ecg_id':str},chunksize=20000):
        sub=ch[ch['fold'].isin([9,10])]
        if len(sub): parts.append(sub)
    return pd.concat(parts,ignore_index=True)
def platt_apply(pl_path,raw):
    try:
        cal=joblib.load(os.path.join(MOD,pl_path))
        try: return cal.predict_proba(np.asarray(raw).reshape(-1,1))[:,1]
        except Exception: return cal.predict(np.asarray(raw))
    except Exception: return np.full(len(raw),np.nan)
from tqdm import tqdm
base=meta[meta.fold.isin([9,10])][['ecg_id','source','fold','label']].copy(); base['ecg_id']=base['ecg_id'].astype(str)
S=base.copy(); t0=time.time()
for name,(mr,fsrc,matrix,pl,ptb_only) in REG.items():
    try:
        feats=feat_list(fsrc); mdl=joblib.load(os.path.join(MOD,mr))
        df=load_fold910(matrix,feats); df['ecg_id']=df['ecg_id'].astype(str)
        miss=[f for f in feats if f not in df.columns]
        if miss: raise RuntimeError('missing %d feats in %s (e.g. %s)'%(len(miss),matrix,miss[:2]))
        raw=mdl.predict_proba(df[feats])[:,1]; df['_raw']=raw; df['_cal']=platt_apply(pl,raw)
        S=S.merge(df[['ecg_id','source','_raw','_cal']].rename(columns={'_raw':name+'_raw','_cal':name+'_cal'}),on=KEY,how='left')
        print('  %-5s inferred (%d rows, %d feats) | %.1f min'%(name,len(df),len(feats),(time.time()-t0)/60))
    except Exception as e:
        print('  %-5s FAILED: %s'%(name,e))
# M7 from CSV
for fold,fn in [(9,'m7v1_fold9_scores.csv'),(10,'m7v1_fold10_scores.csv')]:
    d=pd.read_csv(os.path.join(MOD,'M7_run7',fn),dtype={'ecg_id':str})[['ecg_id','source','m7v1_score']]
    S=S.merge(d.rename(columns={'m7v1_score':'M7_raw_%d'%fold}),on=KEY,how='left')
# M6 from CSV (PTB-only; scored by fix_m6_fold10_scores.py: 6 marquette globals + 4 discovery means)
for fold,fn in [(9,'m6_fold9_scores.csv'),(10,'m6_fold10_scores.csv')]:
    d=pd.read_csv(os.path.join(MOD,'M6_marquette',fn),dtype={'ecg_id':str})[['ecg_id','source','m6_score']]
    S=S.merge(d.rename(columns={'m6_score':'M6_raw_%d'%fold}),on=KEY,how='left')
S['M6_raw']=S['M6_raw_9'].fillna(S['M6_raw_10']); S.drop(columns=['M6_raw_9','M6_raw_10'],inplace=True)
S['M7_raw']=S['M7_raw_9'].fillna(S['M7_raw_10']); S.drop(columns=['M7_raw_9','M7_raw_10'],inplace=True)
S.to_csv(CACHE,index=False); print('cache -> fold10_eval_cache.csv (%d rows)'%len(S))
# ---- fold9 SANITY ONLY (fold9 is validation; safe to look) ----
S9=S[S.fold==9]; rows=[]
for name in ['M1','M2','M3','M4','M5v2','best','M6','M7']:
    col=name+'_raw'
    if col not in S.columns: rows.append((name,'MISSING','','')); continue
    sub=S9.dropna(subset=[col])
    if name=='M6': sub=sub[sub.source.map(is_ptb)]
    if sub[col].notna().sum()>=5 and sub.label.sum()>=1:
        ap=average_precision_score(sub.label,sub[col]); rows.append((name,round(ap,3),REF_F9.get(name),('OK' if abs(ap-REF_F9.get(name,ap))<0.08 else '*** DEVIATION ***')))
    else: rows.append((name,'n/a','',''))
print('\nfold9 AP sanity (inference check — fold9 is validation):')
print(pd.DataFrame(rows,columns=['model','fold9_AP','ref','flag']).to_string(index=False))
print('\nBlock 1 OK. (fold10 scores computed & cached; fold10 NOT evaluated yet.)')

## Block 2 — fold9 standardized evaluation (validation; decides nothing) — PART 1

In [ ]:
# ===================== BLOCK 2 — fold9 standardized evaluation (validation; decides NOTHING) =====================
try: S
except NameError: S=reload_scores()
y_oof18, ens_oof18 = build_oof_ens()
S9=S[S.fold==9].dropna(subset=['M3_raw','M4_raw']).copy()
ens9=ens_of(S9); y9=S9.label.values.astype(int)
m=evaluate_standard(name='ensemble_fold9', y_oof=y_oof18, score_oof=ens_oof18, y_test=y9, score_test=ens9,
    figures_dir=FIG, metrics_dir=METRICS, ap_train=None, ap_oof=OOF_AP,
    extra={'members':CFG['members'],'alpha':ALPHA,'frozen_threshold':THR,'note':'fold9 validation of the frozen ensemble; decides nothing'})
pred=(ens9>=THR).astype(int); tp=int(((pred==1)&(y9==1)).sum()); fp=int(((pred==1)&(y9==0)).sum())
fn=int(((pred==0)&(y9==1)).sum()); tn=int(((pred==0)&(y9==0)).sum()); rec=tp/(tp+fn) if tp+fn else 0.0
print('\n=== ENSEMBLE fold9 @ FROZEN threshold %.4f ==='%THR)
print('  TP %d FP %d FN %d TN %d | recall %.3f | precision %.3f'%(tp,fp,fn,tn,rec,(tp/(tp+fp) if tp+fp else 0.0)))
print('\nComparative fold9 table (per-model AP/AUC; M6=PTB subset):')
print(comp_table(S9,9).to_string(index=False))
print('\nfold9 validates that the frozen threshold generalizes; it changes NOTHING (alpha/threshold already frozen).')
if rec<0.1: print('*** WARNING: near-zero recall at frozen threshold on fold9 -> STOP, do NOT proceed to fold10. ***')
else: print('Frozen threshold gives sane recall on fold9 -> Part 1 validation OK. await confirmation before Part 2.')
print('Block 2 OK.')

# ═══════════════════════════════════════════════════════════════════
# ═══  STOP: do not run below until fold9 is validated  ═══
# ═══════════════════════════════════════════════════════════════════

**Run Blocks 0-2 (PART 1) only. Paste the fold9 comparative table + the ensemble recall at the frozen threshold, then run PART 2 below.** Running Block 3 is the single, irreversible fold10 test-set contact.

## Block 3 — fold10 ATOMIC CONTACT (irreversible) — PART 2

In [ ]:
# ===================== BLOCK 3 — fold10 ATOMIC CONTACT (irreversible; the main result) =====================
try: S
except NameError: S=reload_scores()
try: ens_oof18
except NameError: y_oof18, ens_oof18 = build_oof_ens()
S10=S[S.fold==10].dropna(subset=['M3_raw','M4_raw']).copy()
ens10=ens_of(S10); y10=S10.label.values.astype(int)
m=evaluate_standard(name='ensemble_fold10', y_oof=y_oof18, score_oof=ens_oof18, y_test=y10, score_test=ens10,
    figures_dir=FIG, metrics_dir=METRICS, ap_train=None, ap_oof=OOF_AP,
    extra={'members':CFG['members'],'alpha':ALPHA,'frozen_threshold':THR,'note':'fold10 SINGLE atomic test-set contact'})
pred=(ens10>=THR).astype(int); tp=int(((pred==1)&(y10==1)).sum()); fp=int(((pred==1)&(y10==0)).sum())
fn=int(((pred==0)&(y10==1)).sum()); tn=int(((pred==0)&(y10==0)).sum())
ap=float(average_precision_score(y10,ens10)); auc=float(roc_auc_score(y10,ens10)); lo,hi=_boot_ci(y10,ens10,average_precision_score)
print('\n================ ENSEMBLE fold10 — HEADLINE ================')
print('  AP %.4f CI95[%.4f,%.4f] | AUC %.4f | @frozen thr %.4f: TP %d FP %d FN %d TN %d | recall %.3f precision %.3f'%(
    ap,lo,hi,auc,THR,tp,fp,fn,tn,(tp/(tp+fn) if tp+fn else 0),(tp/(tp+fp) if tp+fp else 0)))
print('\nComparative fold10 table (per-model AP/AUC; descriptive, no re-decision; M6=PTB subset):')
comp=comp_table(S10,10); print(comp.to_string(index=False))
S10=S10.assign(ens=ens10,flag=pred)
fnr=S10[(S10.label==1)&(S10.flag==0)]; fpr=S10[(S10.label==0)&(S10.flag==1)]
print('\nEnsemble fold10 FN (missed WPW), with M7 cross-reference:')
cols=[c for c in ['ecg_id','source','ens','M3_raw','M4_raw','M7_raw'] if c in S10.columns]
print(fnr[cols].round(3).to_string(index=False) if len(fnr) else '  (none)')
print('\nEnsemble fold10 FP at frozen threshold: %d (of %d negatives)'%(len(fpr),int((y10==0).sum())))
if len(fpr): print(fpr[[c for c in ['ecg_id','source','ens'] if c in S10.columns]].round(3).head(15).to_string(index=False))
S10.to_csv(os.path.join(METRICS,'fold10_reference_scores.csv'),index=False)
print('\nsaved: reports/metrics/ensemble_fold10_metrics.json (via evaluate_standard) + fold10_reference_scores.csv')
print('Block 3 OK. fold10 is now CONSUMED.')

## Block 3b — per-model standardized fold10 cards (reporting on cached scores) — PART 2

**fold10 is already CONSUMED by Block 3.** This block does NOT touch the test set again: it re-reports the already-frozen per-model fold10 scores from `fold10_reference_scores.csv`, giving each model its own `evaluate_standard` card with **its OWN OOF (folds 1-8) F1-max threshold** (not the ensemble's). No re-inference, no refit, no re-contact.

**Statistical-honesty note (read before over-interpreting the table).** With only **14 WPW in fold10**, every AP confidence interval is very wide and they **overlap heavily**. The ordering that comes out (M7 highest ≈0.745, ensemble ≈0.595, the individual detectors ≈0.55) is **NOT statistically separable** — the differences are within noise at this sample size. The master table below should be read as *"all detectors land in the same AP band on a 14-positive test fold,"* **not** as a ranking of model quality. The unbiased headline result is the frozen ensemble's fold10 AP (Block 3); the per-model cards are descriptive context for the paper's results section.

In [ ]:
# ===================== BLOCK 3b — per-model standardized fold10 cards (reporting on CACHED scores) =====================
# fold10 is CONSUMED. This ONLY re-reports the ALREADY-FROZEN fold10 scores per model, each with its OWN OOF F1-max.
# NO re-inference, NO re-contact, NO refit. Each model's threshold comes from ITS OWN OOF (folds 1-8), not the ensemble's.
import glob
from sklearn.metrics import precision_recall_curve
# capture the ensemble result from Block 3 FIRST (before any local reuse below)
_ens_row=None
try:
    _ens_row=dict(model='ENSEMBLE(M3+M4)',OOF_AP=round(float(OOF_AP),3),fold9_AP=np.nan,
        fold10_AP=round(float(ap),3),fold10_CI='[%.3f,%.3f]'%(lo,hi),fold10_AUC=round(float(auc),3),
        recall_ownF1=round(tp/(tp+fn),3) if (tp+fn) else 0.0,
        precision_ownF1=round(tp/(tp+fp),3) if (tp+fp) else 0.0)
except NameError:
    print('  (ensemble Block-3 vars not in memory -> run Block 3 first; ensemble row skipped)')
cache=reload_scores()                                                                   # fold9 + fold10, {model}_raw
ref10=pd.read_csv(os.path.join(METRICS,'fold10_reference_scores.csv'),dtype={'ecg_id':str})  # fold10 only (frozen)
OOF_FILES={'M1':'m1_combined_oof.csv','M2':'m2_combined_oof.csv','M3':'m3_combined_oof.csv',
           'M4':'m4_combined_oof.csv','M5v2':'m5v2_combined_oof.csv','best':'bestmodel_oof.csv',
           'M7':'m7v1_combined_oof.csv','M6':'m6_combined_oof.csv'}
def _load_oof(key):
    cand=os.path.join(PROC,OOF_FILES.get(key,''))
    if (not OOF_FILES.get(key)) or (not os.path.exists(cand)):
        pats=[key.lower(),key.lower().replace('v2',''),'bestmodel' if key=='best' else key.lower()]
        g=[]
        for p in pats: g+=glob.glob(os.path.join(PROC,'*%s*oof*.csv'%p))
        g=sorted(set(g))
        if not g: return None
        cand=g[0]
    d=pd.read_csv(cand,dtype={'ecg_id':str})
    if 'fold' in d.columns: d=d[d.fold.isin(range(1,9))]
    scol='proba_raw' if 'proba_raw' in d.columns else next((c for c in d.columns if ('raw' in c or 'proba' in c or 'score' in c)),None)
    if scol is None or 'label' not in d.columns: return None
    return d['label'].values.astype(int), d[scol].values, os.path.basename(cand)
def _f1max_thr(y,s):
    p,r,t=precision_recall_curve(y,s); f1=2*p*r/(p+r+1e-12)
    return float(t[int(np.nanargmax(f1[:-1]))]) if len(t) else 0.5
_rows=[]
for _k in ['M1','M2','M3','M4','M5v2','M7','best','M6']:
    _col=_k+'_raw'
    if _col not in ref10.columns: print('  %-6s: no fold10 column, skipped'%_k); continue
    _o=_load_oof(_k)
    if _o is None: print('  %-6s: no usable OOF file, skipped'%_k); continue
    _yo,_so,_ofn=_o
    _f10=ref10.dropna(subset=[_col]).copy()
    if _k=='M6': _f10=_f10[_f10.source.map(is_ptb)]
    _y=_f10.label.values.astype(int); _s=_f10[_col].values
    if _y.sum()<1 or len(_y)<5: print('  %-6s: insufficient fold10 positives, skipped'%_k); continue
    _apo=float(average_precision_score(_yo,_so))
    evaluate_standard(name='%s_fold10'%_k, y_oof=_yo, score_oof=_so, y_test=_y, score_test=_s,
        figures_dir=FIG, metrics_dir=METRICS, ap_oof=_apo,
        extra={'note':'per-model fold10 card; reporting on frozen cached scores; threshold = own OOF F1-max',
               'oof_file':_ofn,'ptb_only':(_k=='M6')})
    _thr=_f1max_thr(_yo,_so); _pred=(_s>=_thr).astype(int)
    _tp=int(((_pred==1)&(_y==1)).sum()); _fp=int(((_pred==1)&(_y==0)).sum()); _fn=int(((_pred==0)&(_y==1)).sum())
    _rec=_tp/(_tp+_fn) if (_tp+_fn) else 0.0; _prec=_tp/(_tp+_fp) if (_tp+_fp) else 0.0
    _ap=float(average_precision_score(_y,_s)); _auc=float(roc_auc_score(_y,_s)); _lo,_hi=_boot_ci(_y,_s,average_precision_score)
    _c9=cache[cache.fold==9].dropna(subset=[_col]).copy()
    if _k=='M6': _c9=_c9[_c9.source.map(is_ptb)]
    _ap9=float(average_precision_score(_c9.label,_c9[_col])) if _c9.label.sum()>=1 else np.nan
    _rows.append(dict(model=_k,OOF_AP=round(_apo,3),fold9_AP=round(_ap9,3),fold10_AP=round(_ap,3),
                      fold10_CI='[%.3f,%.3f]'%(_lo,_hi),fold10_AUC=round(_auc,3),
                      recall_ownF1=round(_rec,3),precision_ownF1=round(_prec,3)))
    print('  %-6s card saved: %s_fold10_metrics.json + %s_fold10_evaluation.png  (OOF=%s)'%(_k,_k,_k,_ofn))
# ---- consolidated master table (every model, all three folds, same protocol) ----
_tab=pd.DataFrame(_rows)
if _ens_row is not None: _tab=pd.concat([_tab,pd.DataFrame([_ens_row])],ignore_index=True)
_tab=_tab[['model','OOF_AP','fold9_AP','fold10_AP','fold10_CI','fold10_AUC','recall_ownF1','precision_ownF1']]
_tab=_tab.sort_values('fold10_AP',ascending=False,na_position='last').reset_index(drop=True)
print('\n'+'='*90)
print('MASTER fold10 COMPARISON  (per-model own-OOF F1-max threshold; ensemble = frozen 0.9969)')
print('='*90)
print(_tab.to_string(index=False))
_tab.to_csv(os.path.join(METRICS,'fold10_master_comparison.csv'),index=False)
print('\nsaved: reports/metrics/fold10_master_comparison.csv  +  {model}_fold10_metrics.json/.png per model')
print('Block 3b OK. (No re-inference, no re-contact — reporting on frozen fold10 scores only.)')

## Block 4 — final summary + freeze note — PART 2

In [ ]:
# ===================== BLOCK 4 — final summary + freeze note =====================
print('='*66); print('DEPLOYED ENSEMBLE (M3+M4 rank-vote, alpha=0.50) — fold10 TEST-SET RESULT'); print('='*66)
print('  fold10 AP  %.4f  CI95[%.4f,%.4f]'%(ap,lo,hi))
print('  fold10 AUC %.4f'%auc)
print('  @frozen threshold %.4f : TP %d FP %d FN %d TN %d (recall %.3f, precision %.3f)'%(
    THR,tp,fp,fn,tn,(tp/(tp+fn) if tp+fn else 0),(tp/(tp+fp) if tp+fp else 0)))
print('  reference OOF AP (folds1-8) was %.4f — fold10 is the unbiased test estimate.'%OOF_AP)
print('\nfold10 is now CONSUMED. No further model changes, no re-tuning, no second contact.')
print('Block 4 OK.')